# Weather Intelligence: Training Model + MLflow
Notebook ini melatih model regresi suhu dan klasifikasi probabilitas hujan untuk horizon 1, 3, 6, dan 24 jam. Model linear diekspor ke `weather_model.json` agar inferensi dapat dijalankan langsung di Google Apps Script tanpa server Python.

**Catatan metodologi:** data historis Open-Meteo berbasis reanalysis, bukan seluruhnya pengukuran stasiun langsung. Split train/test dilakukan berdasarkan waktu. Lakukan evaluasi tambahan untuk beberapa kota sebelum menyimpulkan model layak.

In [1]:
!pip -q install mlflow scikit-learn pandas numpy requests

import json, os, math, time, zipfile, warnings
from pathlib import Path
from datetime import date, timedelta, datetime, timezone
import numpy as np
import pandas as pd
import requests
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, roc_auc_score, brier_score_loss,
                             average_precision_score, log_loss, precision_recall_fscore_support, accuracy_score)
import mlflow
try:
    from google.colab import files
except ImportError:
    files = None
warnings.filterwarnings('ignore', category=FutureWarning)

# ======================= KONFIGURASI =======================
LAT, LON = -7.2575, 112.7521          # Surabaya
START_DATE = '2022-01-01'
END_DATE = (date.today() - timedelta(days=7)).isoformat()   # arsip ERA5 tertunda beberapa hari
HORIZONS = [1, 3, 6, 24]
MODEL_VERSION = 'weather-v2'
RAIN_THRESHOLD_MM = 0.1               # hujan = presipitasi jam target >= 0,1 mm
TEST_FRAC, VAL_FRAC = 0.20, 0.15      # test = 20% terakhir (sama dengan v1); validation = 15% sebelum test
REFIT_ALL_FOR_DEPLOY = True           # model yang diekspor dilatih ulang dengan SEMUA data (hiperparameter tetap)
N_TREES = 120
SEED = 42
CHUNK_CHARS = 40000                   # batas sel Google Sheets = 50.000 karakter

BASE = Path(os.environ.get('WI_BASE', '/content'))
OUT = BASE / 'weather_intelligence_artifacts_v2'
OUT.mkdir(parents=True, exist_ok=True)
MLFLOW_DB = BASE / 'mlflow.db'
MLFLOW_ARTIFACTS = BASE / 'mlartifacts'
MLFLOW_ARTIFACTS.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri('sqlite:///' + str(MLFLOW_DB.resolve()))   # hasil: sqlite:////content/mlflow.db
EXP_NAME = 'weather-intelligence-dashboard-v2'
if mlflow.get_experiment_by_name(EXP_NAME) is None:
    mlflow.create_experiment(EXP_NAME, artifact_location=MLFLOW_ARTIFACTS.resolve().as_uri())
mlflow.set_experiment(EXP_NAME)
print('Tracking URI:', mlflow.get_tracking_uri(), '| artifact root:', mlflow.get_experiment_by_name(EXP_NAME).artifact_location)

Tracking URI: sqlite:////content/mlflow.db | artifact root: file:///content/mlartifacts


## 1. Unduh data historis per jam

In [2]:
# Open-Meteo: gratis untuk penggunaan nonkomersial, atribusi CC BY 4.0 wajib ditampilkan di dashboard.
URL = 'https://archive-api.open-meteo.com/v1/archive'
HOURLY_VARS = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'pressure_msl', 'wind_speed_10m', 'cloud_cover']

def fetch_archive():
    params = {'latitude': LAT, 'longitude': LON, 'start_date': START_DATE, 'end_date': END_DATE,
              'hourly': ','.join(HOURLY_VARS), 'timezone': 'auto', 'wind_speed_unit': 'kmh'}
    last = None
    for attempt in range(3):
        try:
            r = requests.get(URL, params=params, timeout=120)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last = e
            time.sleep(3 * (attempt + 1))
    raise RuntimeError(f'Gagal mengunduh data Open-Meteo: {last}')

payload = fetch_archive()
if 'hourly' not in payload:
    raise RuntimeError('Respons API tidak memiliki bagian hourly: ' + str(payload)[:500])

raw = pd.DataFrame(payload['hourly'])
raw['time'] = pd.to_datetime(raw['time'])
raw = raw.sort_values('time').drop_duplicates('time').set_index('time')
full_index = pd.date_range(raw.index.min(), raw.index.max(), freq='h')   # grid per jam lengkap: lag selalu = jam sebenarnya
df = raw.reindex(full_index).rename_axis('time').reset_index()
for c in HOURLY_VARS:
    df[c] = pd.to_numeric(df[c], errors='coerce')
n_missing = int(df[HOURLY_VARS].isna().any(axis=1).sum())
print('Rows:', len(df), '| Start:', df.time.min(), '| End:', df.time.max(), '| jam dengan nilai kosong:', n_missing)
print('Timezone API:', payload.get('timezone'), '| UTC offset (detik):', payload.get('utc_offset_seconds'))
if len(df) < 5000:
    raise RuntimeError('Data terlalu sedikit. Perluas START_DATE atau cek respons API.')
df.to_csv(OUT / 'raw_hourly.csv', index=False)   # simpan agar eksperimen dapat direproduksi

Rows: 41808 | Start: 2022-01-01 00:00:00 | End: 2026-10-08 23:00:00 | jam dengan nilai kosong: 0
Timezone API: Asia/Jakarta | UTC offset (detik): 25200


## 2. Feature engineering
Urutan fitur di bawah harus sama persis dengan fungsi inference pada Code.gs.

In [3]:
FEATURES_V1 = [
 'temperature_2m_current', 'relative_humidity_2m_current', 'pressure_msl_current',
 'wind_speed_10m_current', 'precipitation_current', 'cloud_cover_current',
 'temperature_lag_1', 'temperature_lag_3', 'temperature_lag_6', 'temperature_lag_24',
 'humidity_lag_1', 'humidity_lag_3', 'humidity_lag_6', 'humidity_lag_24',
 'precipitation_lag_1', 'precipitation_lag_3', 'precipitation_lag_6', 'precipitation_lag_24',
 'pressure_lag_1', 'pressure_lag_24', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos'
]
FEATURES_EXTRA = [
 'pressure_change_3', 'pressure_change_6', 'temperature_change_3', 'humidity_change_3',
 'precip_sum_3', 'precip_sum_6', 'precip_sum_24',
 'temperature_mean_24', 'temperature_range_24', 'humidity_mean_6', 'cloud_mean_6', 'wind_mean_6',
 'dewpoint_depression_current', 'hours_since_rain'
]
FEATURES_V2 = FEATURES_V1 + FEATURES_EXTRA
MIN_HISTORY_HOURS = 25   # indeks terbaru - 24 harus ada

def _lag(a, k):
    out = np.full(len(a), np.nan)
    out[k:] = a[:-k]
    return out

def _roll(a, w, fn):
    out = np.full(len(a), np.nan)
    if len(a) >= w:
        out[w - 1:] = fn(sliding_window_view(a, w), axis=1)
    return out

def _dewpoint(T, RH):   # rumus Magnus (a=17,62; b=243,12 degC); RH di-clip ke [1,100]
    g = np.log(np.clip(RH, 1, 100) / 100.0) + 17.62 * T / (243.12 + T)
    return 243.12 * g / (17.62 - g)

def _hours_since_rain(R, w=24):
    out = np.full(len(R), np.nan)
    if len(R) >= w:
        W = sliding_window_view(R, w)
        rev = (W >= RAIN_THRESHOLD_MM)[:, ::-1]
        hs = np.where(rev.any(axis=1), rev.argmax(axis=1), w).astype(float)
        hs[np.isnan(W).any(axis=1)] = np.nan
        out[w - 1:] = hs
    return out

def build_features(d):
    """d: DataFrame berurutan per jam dengan kolom time + HOURLY_VARS. Mengembalikan fitur untuk setiap baris."""
    T = d['temperature_2m'].to_numpy(float); H = d['relative_humidity_2m'].to_numpy(float)
    P = d['pressure_msl'].to_numpy(float);   W = d['wind_speed_10m'].to_numpy(float)
    C = d['cloud_cover'].to_numpy(float);    R = d['precipitation'].to_numpy(float)
    f = {'temperature_2m_current': T, 'relative_humidity_2m_current': H, 'pressure_msl_current': P,
         'wind_speed_10m_current': W, 'precipitation_current': R, 'cloud_cover_current': C}
    for k in (1, 3, 6, 24):
        f[f'temperature_lag_{k}'] = _lag(T, k); f[f'humidity_lag_{k}'] = _lag(H, k); f[f'precipitation_lag_{k}'] = _lag(R, k)
    f['pressure_lag_1'] = _lag(P, 1); f['pressure_lag_24'] = _lag(P, 24)
    hour = d['time'].dt.hour.to_numpy(); doy = d['time'].dt.dayofyear.to_numpy()
    f['hour_sin'] = np.sin(2 * np.pi * hour / 24); f['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    f['day_of_year_sin'] = np.sin(2 * np.pi * doy / 365.25); f['day_of_year_cos'] = np.cos(2 * np.pi * doy / 365.25)
    f['pressure_change_3'] = P - _lag(P, 3); f['pressure_change_6'] = P - _lag(P, 6)
    f['temperature_change_3'] = T - _lag(T, 3); f['humidity_change_3'] = H - _lag(H, 3)
    f['precip_sum_3'] = _roll(R, 3, np.sum); f['precip_sum_6'] = _roll(R, 6, np.sum); f['precip_sum_24'] = _roll(R, 24, np.sum)
    f['temperature_mean_24'] = _roll(T, 24, np.mean)
    f['temperature_range_24'] = _roll(T, 24, np.max) - _roll(T, 24, np.min)
    f['humidity_mean_6'] = _roll(H, 6, np.mean); f['cloud_mean_6'] = _roll(C, 6, np.mean); f['wind_mean_6'] = _roll(W, 6, np.mean)
    f['dewpoint_depression_current'] = T - _dewpoint(T, H)
    f['hours_since_rain'] = _hours_since_rain(R)
    return pd.DataFrame(f, index=d.index)[FEATURES_V2]

FEAT = build_features(df)
print('Fitur v1:', len(FEATURES_V1), '| fitur v2:', len(FEATURES_V2))
display(FEAT.tail(3).T.head(12))

Fitur v1: 24 | fitur v2: 38


,41805,41806,41807
temperature_2m_current,28.9,28.7,28.7
relative_humidity_2m_current,72.0,70.0,68.0
pressure_msl_current,1012.5,1012.4,1012.0
wind_speed_10m_current,14.8,12.9,9.4
precipitation_current,0.0,0.0,0.0
cloud_cover_current,91.0,98.0,98.0
temperature_lag_1,29.2,28.9,28.7
temperature_lag_3,30.2,29.7,29.2
temperature_lag_6,32.2,31.8,30.7
temperature_lag_24,28.8,28.6,28.5


## 3. Train/test split berdasarkan waktu, training, dan logging MLflow
Target klasifikasi hujan didefinisikan sebagai curah hujan pada jam target >= 0,1 mm. Ini adalah probabilitas hujan pada jam target, bukan probabilitas kumulatif selama seluruh horizon.

In [4]:
n = len(df)
cut_val = df.loc[int(n * (1 - TEST_FRAC - VAL_FRAC)), 'time']
cut_test = df.loc[int(n * (1 - TEST_FRAC)), 'time']
print('cut_val:', cut_val, '| cut_test:', cut_test)

def make_xy(h):
    y_abs = df['temperature_2m'].shift(-h); y_prec = df['precipitation'].shift(-h); t_tgt = df['time'].shift(-h)
    ok = FEAT.notna().all(axis=1) & y_abs.notna() & y_prec.notna() & t_tgt.notna()
    d = pd.DataFrame({
        't_target': t_tgt[ok], 'target_hour': t_tgt[ok].dt.hour,
        'T0': df['temperature_2m'][ok], 'y_abs': y_abs[ok], 'y_delta': (y_abs - df['temperature_2m'])[ok],
        'T_diurnal': df['temperature_2m'].shift(24 - h)[ok],          # suhu pada waktu target - 24 jam (sudah diketahui saat t)
        'y_rain': (y_prec[ok] >= RAIN_THRESHOLD_MM).astype(int)})
    d['split'] = np.where(d['t_target'] < cut_val, 'train', np.where(d['t_target'] < cut_test, 'val', 'test'))
    return d, FEAT.loc[ok]

class Fitted:
    """Pembungkus model yang dapat diekspor ke JSON untuk Apps Script."""
    def __init__(self, kind, feats, params, task):
        self.kind, self.feats, self.params, self.task = kind, list(feats), dict(params), task
    def fit(self, X, y):
        Xf = X[self.feats]
        if self.kind in ('ridge', 'logit'):
            self.scaler = StandardScaler().fit(Xf.to_numpy())
            Z = self.scaler.transform(Xf.to_numpy())
            if self.kind == 'ridge':
                self.model = Ridge(alpha=self.params['alpha'])
            else:
                self.model = LogisticRegression(C=self.params.get('C', 1.0), class_weight=self.params.get('class_weight'), max_iter=5000)
            self.model.fit(Z, y)
        else:
            cls = GradientBoostingRegressor if self.task == 'temp' else GradientBoostingClassifier
            self.model = cls(random_state=SEED, **self.params)
            self.model.fit(Xf.to_numpy(), y)
        if self.task == 'rain':
            assert list(self.model.classes_) == [0, 1], 'Label hujan harus memiliki dua kelas'
        self._x0 = Xf.to_numpy()[:1].copy()
        return self
    def predict(self, X):
        Xf = X[self.feats].to_numpy()
        if self.kind in ('ridge', 'logit'):
            Z = self.scaler.transform(Xf)
            return self.model.predict(Z) if self.kind == 'ridge' else self.model.predict_proba(Z)[:, 1]
        return self.model.predict(Xf) if self.task == 'temp' else self.model.predict_proba(Xf)[:, 1]
    def export(self):
        if self.kind in ('ridge', 'logit'):
            coef = self.model.coef_.ravel(); mu, sd = self.scaler.mean_, self.scaler.scale_
            icpt = float(np.ravel(self.model.intercept_)[0]) - float(np.sum(coef * mu / sd))   # lebur scaler ke koefisien
            return {'type': 'linear', 'features': self.feats, 'coef': (coef / sd).astype(float).tolist(), 'intercept': icpt}
        est = self.model.estimators_[:, 0]
        lr = float(self.model.learning_rate)
        x0 = self._x0.astype(np.float32)
        raw0 = self.model.predict(x0)[0] if self.task == 'temp' else self.model.decision_function(x0)[0]
        init = float(raw0 - lr * sum(float(e.predict(x0)[0]) for e in est))
        trees = []
        for e in est:
            t = e.tree_
            trees.append({'l': t.children_left.tolist(), 'r': t.children_right.tolist(), 'f': t.feature.tolist(),
                          't': t.threshold.tolist(), 'v': np.round(t.value[:, 0, 0], 8).tolist()})
        return {'type': 'gbdt', 'features': self.feats, 'init': init, 'learning_rate': lr, 'trees': trees}

def _ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1); e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)

def temp_scores(d_sub, pred_abs):
    y = d_sub['y_abs'].to_numpy()
    return {'MAE_C': float(mean_absolute_error(y, pred_abs)), 'RMSE_C': float(np.sqrt(mean_squared_error(y, pred_abs)))}

def rain_scores(y, p, thr=0.5):
    y = np.asarray(y); p = np.asarray(p, float)
    out = {'Brier': float(brier_score_loss(y, p)), 'LogLoss': float(log_loss(y, np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1]))}
    try:
        out['ROC_AUC'] = float(roc_auc_score(y, p)); out['PR_AUC'] = float(average_precision_score(y, p))
    except ValueError:
        out['ROC_AUC'] = out['PR_AUC'] = float('nan')
    yhat = (p >= thr).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    out.update({'precision': float(pr), 'recall': float(rc), 'F1': float(f1), 'accuracy': float(accuracy_score(y, yhat)), 'ECE': _ece(y, p)})
    return out

def best_f1_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    f1s = [precision_recall_fscore_support(y, (p >= t).astype(int), average='binary', zero_division=0)[2] for t in grid]
    return float(grid[int(np.argmax(f1s))])

GB = dict(learning_rate=0.05, n_estimators=N_TREES, subsample=0.8, min_samples_leaf=50)
TEMP_CANDS = {
    'ridge_v1_a1':   ('ridge', FEATURES_V1, {'alpha': 1.0}),
    'ridge_v1_a30':  ('ridge', FEATURES_V1, {'alpha': 30.0}),
    'ridge_v2_a1':   ('ridge', FEATURES_V2, {'alpha': 1.0}),
    'ridge_v2_a30':  ('ridge', FEATURES_V2, {'alpha': 30.0}),
    'ridge_v2_a300': ('ridge', FEATURES_V2, {'alpha': 300.0}),
    'gbdt_v2_d3':    ('gbdt', FEATURES_V2, dict(GB, max_depth=3)),
    'gbdt_v2_d4':    ('gbdt', FEATURES_V2, dict(GB, max_depth=4)),
}
RAIN_CANDS = {
    'logit_v1_balanced': ('logit', FEATURES_V1, {'C': 1.0, 'class_weight': 'balanced'}),   # = konfigurasi v1
    'logit_v1':          ('logit', FEATURES_V1, {'C': 1.0}),
    'logit_v2_C0.1':     ('logit', FEATURES_V2, {'C': 0.1}),
    'logit_v2_C1':       ('logit', FEATURES_V2, {'C': 1.0}),
    'gbdt_v2_d3':        ('gbdt', FEATURES_V2, dict(GB, max_depth=3)),
    'gbdt_v2_d4':        ('gbdt', FEATURES_V2, dict(GB, max_depth=4)),
}
REF_TEMP, REF_RAIN = 'ridge_v1_a1', 'logit_v1_balanced'

cut_val: 2025-02-06 07:00:00 | cut_test: 2025-10-25 14:00:00


## 4. Training, seleksi (validation), dan evaluasi akhir (test) + logging MLflow

In [5]:
val_rows, test_rows, base_rows, calib_rows = [], [], [], []
deployed, selected, hsum = {}, {}, {}

def fit_pred(name, cands, task, X, d, fit_mask, pred_mask):
    kind, feats, params = cands[name]
    m = Fitted(kind, feats, params, task).fit(X.loc[fit_mask], d.loc[fit_mask, 'y_delta' if task == 'temp' else 'y_rain'])
    return m, m.predict(X.loc[pred_mask])

with mlflow.start_run(run_name='weather-v2-multi-horizon') as parent:
    parent_run_id = parent.info.run_id
    mlflow.log_params({'latitude': LAT, 'longitude': LON, 'start_date': START_DATE, 'end_date': END_DATE,
                       'cut_val': str(cut_val), 'cut_test': str(cut_test), 'rain_threshold_mm': RAIN_THRESHOLD_MM,
                       'n_trees': N_TREES, 'refit_all_for_deploy': REFIT_ALL_FOR_DEPLOY, 'model_version': MODEL_VERSION})
    for h in HORIZONS:
        d, X = make_xy(h)
        sp = d['split'].to_numpy()
        tr, va, te = (sp == 'train'), (sp == 'val'), (sp == 'test')
        trva = tr | va; allm = np.ones(len(d), bool)
        if tr.sum() < 500 or va.sum() < 100 or te.sum() < 100:
            raise RuntimeError(f'Data terlalu sedikit untuk horizon {h}')
        with mlflow.start_run(run_name=f'h{h}', nested=True):
            # ---------- seleksi suhu (validation) ----------
            best_t, best_t_mae = None, np.inf
            for name in TEMP_CANDS:
                _, dp = fit_pred(name, TEMP_CANDS, 'temp', X, d, tr, va)
                s = temp_scores(d[va], d.loc[va, 'T0'].to_numpy() + dp)
                val_rows.append({'horizon_hours': h, 'target': 'temperature', 'model': name, **s})
                if s['MAE_C'] < best_t_mae: best_t, best_t_mae = name, s['MAE_C']
            # ---------- seleksi hujan (validation) ----------
            best_r, best_r_brier, val_probs = None, np.inf, {}
            for name in RAIN_CANDS:
                _, pp = fit_pred(name, RAIN_CANDS, 'rain', X, d, tr, va)
                val_probs[name] = pp
                s = rain_scores(d.loc[va, 'y_rain'], pp)
                val_rows.append({'horizon_hours': h, 'target': 'rain', 'model': name, **s})
                if s['Brier'] < best_r_brier: best_r, best_r_brier = name, s['Brier']
            thr = best_f1_threshold(d.loc[va, 'y_rain'].to_numpy(), val_probs[best_r])
            selected[h] = {'temperature': best_t, 'rain': best_r, 'rain_threshold_f1': thr}

            # ---------- evaluasi test (sekali): refit train+val ----------
            T0_te = d.loc[te, 'T0'].to_numpy()
            def trow(name, pred_abs, **kw):
                test_rows_local = temp_scores(d[te], pred_abs); return {'horizon_hours': h, 'target': 'temperature', 'model': name, **test_rows_local, **kw}
            base_rows.append(trow('persistence', T0_te))
            base_rows.append(trow('diurnal_persistence', d.loc[te, 'T_diurnal'].to_numpy()))
            m_ref, dp_ref = fit_pred(REF_TEMP, TEMP_CANDS, 'temp', X, d, trva, te)
            base_rows.append(trow(f'{REF_TEMP} (gaya v1)', T0_te + dp_ref))
            m_t, dp_t = fit_pred(best_t, TEMP_CANDS, 'temp', X, d, trva, te)
            t_sel = temp_scores(d[te], T0_te + dp_t)
            base_rows.append(trow(f'{best_t} (TERPILIH)', T0_te + dp_t))

            y_te = d.loc[te, 'y_rain'].to_numpy()
            base_rate = float(d.loc[trva, 'y_rain'].mean())
            hr = d.loc[trva].groupby('target_hour')['y_rain'].mean()
            p_clim = np.full(te.sum(), base_rate); p_hour = d.loc[te, 'target_hour'].map(hr).fillna(base_rate).to_numpy()
            def rrow(name, p, **kw): return {'horizon_hours': h, 'target': 'rain', 'model': name, **rain_scores(y_te, p, thr), **kw}
            base_rows.append(rrow('klimatologi_konstan', p_clim))
            base_rows.append(rrow('klimatologi_per_jam', p_hour))
            m_rr, pp_ref = fit_pred(REF_RAIN, RAIN_CANDS, 'rain', X, d, trva, te)
            base_rows.append(rrow(f'{REF_RAIN} (gaya v1)', pp_ref))
            m_r, pp_sel = fit_pred(best_r, RAIN_CANDS, 'rain', X, d, trva, te)
            base_rows.append(rrow(f'{best_r} (TERPILIH)', pp_sel))
            r_sel = rain_scores(y_te, pp_sel, 0.5); r_thr = rain_scores(y_te, pp_sel, thr)
            brier_clim = brier_score_loss(y_te, p_clim)
            mae_pers = mean_absolute_error(d.loc[te, 'y_abs'], T0_te)
            mae_diur = mean_absolute_error(d.loc[te, 'y_abs'], d.loc[te, 'T_diurnal'])
            edges = np.linspace(0, 1, 11); bidx = np.clip(np.digitize(pp_sel, edges[1:-1]), 0, 9)
            for b in range(10):
                mk = bidx == b
                if mk.any(): calib_rows.append({'horizon_hours': h, 'bin': b, 'n': int(mk.sum()), 'mean_pred': float(pp_sel[mk].mean()), 'obs_freq': float(y_te[mk].mean())})
            row = {'horizon_hours': h, 'train_rows': int(trva.sum()), 'test_rows': int(te.sum()),
                   'temperature_MAE_C': t_sel['MAE_C'], 'temperature_RMSE_C': t_sel['RMSE_C'],
                   'rain_Brier_score': r_sel['Brier'], 'rain_ROC_AUC': r_sel['ROC_AUC'], 'rain_accuracy_at_0_5': r_sel['accuracy'],
                   'temp_model': best_t, 'rain_model': best_r,
                   'MAE_skill_vs_persistence': 1 - t_sel['MAE_C'] / mae_pers, 'MAE_skill_vs_diurnal': 1 - t_sel['MAE_C'] / mae_diur,
                   'Brier_skill_vs_climatology': 1 - r_sel['Brier'] / brier_clim, 'rain_PR_AUC': r_sel['PR_AUC'],
                   'rain_threshold_F1_val': thr, 'rain_precision_at_thr': r_thr['precision'], 'rain_recall_at_thr': r_thr['recall'],
                   'rain_F1_at_thr': r_thr['F1'], 'rain_ECE': r_sel['ECE'], 'rain_base_rate_test': float(y_te.mean())}
            test_rows.append(row)
            mlflow.log_params({'temp_model': best_t, 'rain_model': best_r})
            mlflow.log_metrics({k: v for k, v in row.items() if isinstance(v, (int, float)) and np.isfinite(v) and k != 'horizon_hours'})

            # ---------- model untuk deployment ----------
            dep_mask = allm if REFIT_ALL_FOR_DEPLOY else trva
            md_t = Fitted(*TEMP_CANDS[best_t], 'temp').fit(X.loc[dep_mask], d.loc[dep_mask, 'y_delta'])
            md_r = Fitted(*RAIN_CANDS[best_r], 'rain').fit(X.loc[dep_mask], d.loc[dep_mask, 'y_rain'])
            deployed[h] = {'temp': md_t, 'rain': md_r}
            print(f'h+{h:>2}: suhu={best_t:<14} MAE={t_sel["MAE_C"]:.3f} (persist {mae_pers:.3f}, diurnal {mae_diur:.3f}) | '
                  f'hujan={best_r:<18} Brier={r_sel["Brier"]:.4f} (klim {brier_clim:.4f}) AUC={r_sel["ROC_AUC"]:.3f} PR-AUC={r_sel["PR_AUC"]:.3f}')

    val_df = pd.DataFrame(val_rows); base_df = pd.DataFrame(base_rows); metrics_df = pd.DataFrame(test_rows); calib_df = pd.DataFrame(calib_rows)
    metrics_path = OUT / 'test_metrics.csv'; metrics_df.to_csv(metrics_path, index=False)
    val_df.to_csv(OUT / 'model_selection_validation.csv', index=False)
    base_df.to_csv(OUT / 'baseline_comparison.csv', index=False)
    calib_df.to_csv(OUT / 'rain_calibration.csv', index=False)
    for p in ['test_metrics.csv', 'model_selection_validation.csv', 'baseline_comparison.csv', 'rain_calibration.csv']:
        mlflow.log_artifact(str(OUT / p))
    mlflow.log_artifact(str(OUT / 'raw_hourly.csv'))

display(metrics_df.round(4))
print('Perbandingan dengan baseline (data test):')
display(base_df.round(4))

h+ 1: suhu=gbdt_v2_d4     MAE=0.353 (persist 0.596, diurnal 0.791) | hujan=gbdt_v2_d4         Brier=0.1029 (klim 0.1980) AUC=0.912 PR-AUC=0.791
h+ 3: suhu=gbdt_v2_d4     MAE=0.621 (persist 1.453, diurnal 0.791) | hujan=gbdt_v2_d4         Brier=0.1416 (klim 0.1980) AUC=0.835 PR-AUC=0.639
h+ 6: suhu=gbdt_v2_d4     MAE=0.722 (persist 2.432, diurnal 0.791) | hujan=gbdt_v2_d4         Brier=0.1511 (klim 0.1980) AUC=0.805 PR-AUC=0.602
h+24: suhu=gbdt_v2_d4     MAE=0.724 (persist 0.791, diurnal 0.791) | hujan=gbdt_v2_d3         Brier=0.1554 (klim 0.1980) AUC=0.790 PR-AUC=0.579


,horizon_hours,train_rows,test_rows,temperature_MAE_C,temperature_RMSE_C,rain_Brier_score,rain_ROC_AUC,rain_accuracy_at_0_5,temp_model,rain_model,MAE_skill_vs_persistence,MAE_skill_vs_diurnal,Brier_skill_vs_climatology,rain_PR_AUC,rain_threshold_F1_val,rain_precision_at_thr,rain_recall_at_thr,rain_F1_at_thr,rain_ECE,rain_base_rate_test
0,1,33421,8362,0.3526,0.5275,0.1029,0.9118,0.8560,gbdt_v2_d4,gbdt_v2_d4,0.4080,0.5542,0.4801,0.7914,0.37,0.7223,0.7547,0.7382,0.0194,0.2716
1,3,33419,8362,0.6214,0.8265,0.1416,0.8354,0.7884,gbdt_v2_d4,gbdt_v2_d4,0.5725,0.2144,0.2849,0.6393,0.36,0.5881,0.6552,0.6199,0.0194,0.2716
2,6,33416,8362,0.7224,0.9504,0.1511,0.8055,0.7792,gbdt_v2_d4,gbdt_v2_d4,0.7030,0.0866,0.2366,0.6018,0.31,0.5220,0.6579,0.5821,0.0193,0.2716
3,24,33398,8362,0.7239,0.9829,0.1554,0.7900,0.7771,gbdt_v2_d4,gbdt_v2_d3,0.0847,0.0847,0.2149,0.5787,0.30,0.4928,0.6473,0.5596,0.0260,0.2716


Perbandingan dengan baseline (data test):


,horizon_hours,target,model,MAE_C,RMSE_C,Brier,LogLoss,ROC_AUC,PR_AUC,precision,recall,F1,accuracy,ECE
0,1,temperature,persistence,0.5956,0.8194,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,temperature,diurnal_persistence,0.7909,1.1336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,temperature,ridge_v1_a1 (gaya v1),0.4383,0.6215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,temperature,gbdt_v2_d4 (TERPILIH),0.3526,0.5275,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,rain,klimatologi_konstan,NaN,NaN,0.1980,0.5852,0.5000,0.2716,0.0000,0.0000,0.0000,0.7284,0.0126
5,1,rain,klimatologi_per_jam,NaN,NaN,0.1923,0.5720,0.6116,0.3547,0.4037,0.3100,0.3507,0.6882,0.0225
6,1,rain,logit_v1_balanced (gaya v1),NaN,NaN,0.1349,0.4421,0.8570,0.7175,0.5481,0.7878,0.6464,0.7660,0.0673
7,1,rain,gbdt_v2_d4 (TERPILIH),NaN,NaN,0.1029,0.3306,0.9118,0.7914,0.7223,0.7547,0.7382,0.8546,0.0194
8,3,temperature,persistence,1.4534,1.8738,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,3,temperature,diurnal_persistence,0.7909,1.1336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Ekspor model, parity fixture, dan chunk untuk Google Sheets
Unduh ZIP, ekstrak, lalu salin isi file `weather_model.json` ke sel `ML_Model!A1` pada spreadsheet Apps Script. Jangan menyalin tanda code fence Markdown. File `mlruns` berisi tracking MLflow lokal; simpan ZIP sebagai arsip eksperimen. Karena Colab bersifat sementara, unduh hasil sebelum runtime terputus.

In [14]:

import csv
import json
import zipfile
import numpy as np
import pandas as pd

from datetime import datetime, timezone


# ============================================================
# 1. MEMBENTUK RIWAYAT DAN PARITY FIXTURE
# ============================================================

def window_records(e, size=72):
    """Mengambil riwayat cuaca berurutan selama 72 jam."""
    w = df.iloc[e - size + 1:e + 1].reset_index(drop=True)

    history = []

    for i, t in enumerate(w["time"]):
        record = {
            "time": t.strftime("%Y-%m-%dT%H:%M")
        }

        for col in HOURLY_VARS:
            value = w.loc[i, col]
            record[col] = (
                None if pd.isna(value) else float(value)
            )

        history.append(record)

    return w, history


def expected_for_window(w):
    """Menghasilkan prediksi referensi Python dari jendela terakhir."""
    fw = build_features(w).iloc[[-1]]

    if fw.isna().any(axis=None):
        raise ValueError(
            "Jendela parity fixture mengandung fitur NaN."
        )

    feature_values = fw.to_numpy(dtype=float)

    if not np.isfinite(feature_values).all():
        raise ValueError(
            "Jendela parity fixture mengandung fitur non-finite."
        )

    out = {}

    for h in HORIZONS:
        temp_model = deployed[h]["temp"]
        rain_model = deployed[h]["rain"]

        # Target suhu dilatih sebagai delta T.
        temperature_c = float(
            w["temperature_2m"].iloc[-1]
            + temp_model.predict(fw)[0]
        )

        # predict() untuk model hujan harus menghasilkan
        # probabilitas kelas positif dalam rentang [0, 1].
        rain_probability = float(
            rain_model.predict(fw)[0]
        )

        if not np.isfinite(temperature_c):
            raise ValueError(
                f"Prediksi suhu non-finite pada horizon {h}."
            )

        if (
            not np.isfinite(rain_probability)
            or not 0 <= rain_probability <= 1
        ):
            raise ValueError(
                f"Probabilitas hujan tidak valid pada horizon {h}: "
                f"{rain_probability}"
            )

        out[str(h)] = {
            "temperature_c": temperature_c,
            "rain_probability": rain_probability
        }

    return fw, out


# Pilih tiga jendela:
# 1. Jendela data terakhir.
# 2. Jendela hujan lebat pada periode test, jika tersedia.
# 3. Jendela di tengah periode data.

valid_ends = [
    e
    for e in range(72, len(df))
    if df.iloc[e - 71:e + 1][HOURLY_VARS]
        .notna()
        .all()
        .all()
]

if len(valid_ends) < 3:
    raise RuntimeError(
        "Data tidak cukup untuk membuat tiga parity fixture valid."
    )

rainy = [
    e
    for e in valid_ends
    if (
        df.loc[e, "time"] >= cut_test
        and df.loc[e, "precipitation"] >= 2.0
    )
]

middle = valid_ends[len(valid_ends) // 2]
rainy_or_fallback = (
    rainy[-1] if rainy else valid_ends[-2]
)

fixture_ends = sorted(
    set([
        valid_ends[-1],
        rainy_or_fallback,
        middle
    ])
)

fixtures = []

for e in fixture_ends:
    w, hist = window_records(e)

    fw, exp = expected_for_window(w)

    # Pastikan fitur dari jendela 72 jam sama dengan
    # fitur pada dataset penuh di indeks yang sama.
    full_features = FEAT.loc[e, FEATURES_V2].to_numpy(
        dtype=float
    )

    window_features = fw.iloc[0].to_numpy(dtype=float)

    if not np.allclose(
        window_features,
        full_features,
        rtol=0,
        atol=1e-10,
        equal_nan=False
    ):
        raise AssertionError(
            f"Fitur jendela berbeda dari fitur dataset penuh "
            f"pada indeks {e}."
        )

    # INI PERUBAHAN UTAMA:
    # Simpan 38 fitur referensi yang dihitung oleh Python.
    feature_map = {
        name: float(fw.iloc[0][name])
        for name in FEATURES_V2
    }

    if not all(
        np.isfinite(value)
        for value in feature_map.values()
    ):
        raise ValueError(
            f"Fitur fixture tidak valid pada indeks {e}."
        )

    fixtures.append({
        "history": hist,
        "features": feature_map,
        "expected": exp
    })

print("Fixture dibuat untuk indeks:", fixture_ends)
print("Jumlah fixture:", len(fixtures))
print(
    "Semua fixture memiliki fitur:",
    all("features" in item for item in fixtures)
)


# ============================================================
# 2. MEMBANGUN JSON MODEL
# ============================================================

model_export = {
    "schema_version": 2,
    "model_type": "per_target_ensemble",
    "model_version": MODEL_VERSION,

    "created_utc": datetime.now(timezone.utc).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    ),

    "trained_location": {
        "latitude": LAT,
        "longitude": LON
    },

    "training_period": {
        "start": START_DATE,
        "end": END_DATE
    },

    "timezone": payload.get("timezone"),
    "utc_offset_seconds": payload.get("utc_offset_seconds"),

    "data_source": (
        "Open-Meteo Historical Weather API "
        "(reanalysis), CC BY 4.0"
    ),

    "target_definition": (
        "temperature: T(t+h) = T(t) + delta "
        "(model memprediksi delta); "
        "rain: P(precipitation(t+h) >= rain_threshold_mm) "
        "pada jam target"
    ),

    "rain_threshold_mm": RAIN_THRESHOLD_MM,
    "min_history_hours": MIN_HISTORY_HOURS,
    "feature_names": FEATURES_V2,
    "refit_all_for_deploy": REFIT_ALL_FOR_DEPLOY,

    "horizons": {},

    "evaluation_summary": json.loads(
        metrics_df.round(6).to_json(orient="records")
    ),

    # Setiap fixture sekarang mempunyai history, features, expected.
    "parity_fixture": fixtures
}


# Ekspor model suhu dan hujan untuk setiap horizon.
for h in HORIZONS:
    temp_component = deployed[h]["temp"].export()
    temp_component["model_name"] = selected[h]["temperature"]

    rain_component = deployed[h]["rain"].export()
    rain_component["model_name"] = selected[h]["rain"]
    rain_component["f1_threshold"] = selected[h][
        "rain_threshold_f1"
    ]

    model_export["horizons"][str(h)] = {
        "temperature": temp_component,
        "rain": rain_component
    }


# ============================================================
# 3. VALIDASI STRUKTUR SEBELUM EKSPOR
# ============================================================

assert len(model_export["feature_names"]) == 38, (
    "Jumlah fitur model bukan 38."
)

assert len(model_export["parity_fixture"]) > 0, (
    "Parity fixture kosong."
)

for fixture in model_export["parity_fixture"]:
    assert len(fixture["history"]) == 72
    assert len(fixture["features"]) == 38
    assert set(fixture["expected"].keys()) == {
        "1", "3", "6", "24"
    }

for h in ["1", "3", "6", "24"]:
    for target in ["temperature", "rain"]:
        component = model_export["horizons"][h][target]

        assert component["type"] in ("gbdt", "linear"), (
            f"Tipe model tidak didukung: "
            f"horizon {h}, target {target}"
        )

        if component["type"] == "gbdt":
            assert component.get("trees"), (
                f"Pohon tidak tersedia pada horizon {h}, "
                f"target {target}"
            )

            assert len(component["features"]) == 38, (
                f"Jumlah fitur komponen tidak sesuai pada "
                f"horizon {h}, target {target}"
            )

print("Validasi struktur ekspor berhasil.")


# ============================================================
# 4. MENULIS weather_model.json
# ============================================================

OUT.mkdir(parents=True, exist_ok=True)

model_path = OUT / "weather_model.json"

text = json.dumps(
    model_export,
    separators=(",", ":"),
    allow_nan=False
)

model_path.write_text(
    text,
    encoding="utf-8"
)

print(
    f"weather_model.json: {len(text):,} karakter | "
    f"{len(text.encode('utf-8')):,} byte"
)


# ============================================================
# 5. MENYIMPAN VERSI CHUNKED UNTUK ARSIP
# ============================================================

chunks = [
    text[i:i + CHUNK_CHARS]
    for i in range(0, len(text), CHUNK_CHARS)
]

chunks_path = OUT / "ML_Model_chunks.csv"

with open(
    chunks_path,
    "w",
    newline="",
    encoding="utf-8"
) as fh:
    writer = csv.writer(fh, quoting=csv.QUOTE_ALL)

    for i, chunk in enumerate(chunks, start=1):
        writer.writerow([i, chunk])

print(
    f"ML_Model_chunks.csv: {len(chunks)} chunk | "
    f"maksimum {CHUNK_CHARS:,} karakter per chunk"
)


# ============================================================
# 6. MEMBUAT CATATAN EKSPERIMEN
# ============================================================

note = {
    "timestamp_utc": model_export["created_utc"],
    "model_version": MODEL_VERSION,
    "mlflow_run_id": parent_run_id,
    "training_period": f"{START_DATE}..{END_DATE}",
    "test_from": str(cut_test),

    "selected": json.dumps({
        str(h): {
            k: v
            for k, v in selected[h].items()
        }
        for h in HORIZONS
    }),

    "summary": "; ".join(
        (
            f"h{r['horizon_hours']}: "
            f"MAE {r['temperature_MAE_C']:.2f}C, "
            f"BSS {r['Brier_skill_vs_climatology']:.3f}"
        )
        for r in test_rows
    )
}

notes_path = OUT / "experiment_notes_row.csv"

pd.DataFrame([note]).to_csv(
    notes_path,
    index=False
)


# ============================================================
# 7. LOG ARTEFAK MLflow
# ============================================================

with mlflow.start_run(run_id=parent_run_id):
    mlflow.log_artifact(str(model_path))
    mlflow.log_artifact(str(notes_path))


# ============================================================
# 8. MEMBUAT ZIP ARTEFAK
# ============================================================

bundle = BASE / "weather_intelligence_ml_artifacts_v2.zip"

with zipfile.ZipFile(
    bundle,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:

    for path in sorted(OUT.glob("*")):
        if path.is_file():
            zf.write(path, arcname=path.name)

    if MLFLOW_DB.exists():
        zf.write(
            MLFLOW_DB,
            arcname="mlflow.db"
        )

    for path in MLFLOW_ARTIFACTS.rglob("*"):
        if path.is_file():
            zf.write(
                path,
                arcname=(
                    "mlartifacts/"
                    + str(path.relative_to(MLFLOW_ARTIFACTS))
                )
            )

print(
    "ZIP berhasil dibuat:",
    bundle,
    "| bytes:",
    bundle.stat().st_size
)


# ============================================================
# 9. UNDUH ZIP DI GOOGLE COLAB
# ============================================================

try:
    from google.colab import files as colab_files
    colab_files.download(str(bundle))
except Exception as exc:
    print(
        "ZIP tersimpan di:", bundle,
        "| Unduh manual jika perlu. Detail:", exc
    )

Fixture dibuat untuk indeks: [20940, 38448, 41807]
Jumlah fixture: 3
Semua fixture memiliki fitur: True
Validasi struktur ekspor berhasil.
weather_model.json: 870,807 karakter | 870,807 byte
ML_Model_chunks.csv: 22 chunk | maksimum 40,000 karakter per chunk
ZIP berhasil dibuat: /content/weather_intelligence_ml_artifacts_v2.zip | bytes: 1508060


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Cara membaca hasil
- **Temperature MAE (°C):** rata-rata selisih absolut suhu prediksi dan target. Lebih kecil lebih baik.
- **Temperature RMSE (°C):** memberi penalti lebih besar untuk error ekstrem.
- **Rain Brier score:** mengukur kualitas probabilitas hujan; makin dekat 0 makin baik.
- **Rain ROC-AUC:** kemampuan memeringkat jam hujan lebih tinggi dari jam tanpa hujan; bandingkan dengan baseline.
- Hasil satu koordinat tidak boleh dianggap mewakili seluruh wilayah. Uji beberapa kota, periode, dan baseline. Prakiraan meteorologi Open-Meteo adalah pembanding kuat; model sederhana belum tentu lebih baik.
- Model Apps Script adalah inferensi linear/logistik hasil training. MLflow dipakai untuk melacak eksperimen training; MLflow Tracking Server tidak berjalan di Apps Script.

In [7]:
import json
from pathlib import Path

model_path = Path("/content/weather_intelligence_artifacts_v2/weather_model.json")

raw = model_path.read_text(encoding="utf-8")
model = json.loads(raw)

print("Ukuran teks:", len(raw), "karakter")
print("Ukuran file:", len(raw.encode("utf-8")), "byte")
print("Versi model:", model.get("model_version"))
print("Tipe model:", model.get("model_type"))
print("Fitur:", len(model.get("feature_names", [])))
print("Horizon:", list(model.get("horizons", {}).keys()))

for horizon, models in model.get("horizons", {}).items():
    print(
        "Horizon:", horizon,
        "| Komponen:", list(models.keys())
    )


Ukuran teks: 867447 karakter
Ukuran file: 867447 byte
Versi model: weather-v2
Tipe model: None
Fitur: 38
Horizon: ['1', '3', '6', '24']
Horizon: 1 | Komponen: ['temperature', 'rain']
Horizon: 3 | Komponen: ['temperature', 'rain']
Horizon: 6 | Komponen: ['temperature', 'rain']
Horizon: 24 | Komponen: ['temperature', 'rain']


In [10]:
import json
from pathlib import Path

model_path = Path("/content/weather_intelligence_artifacts_v2/weather_model.json")
model = json.loads(model_path.read_text(encoding="utf-8"))

print("=== METADATA MODEL ===")
print("Top-level keys:", list(model.keys()))
print("Model version:", model.get("model_version"))
print("Model type:", model.get("model_type"))
print("Jumlah fitur:", len(model.get("feature_names", [])))

print("\n=== DAFTAR FITUR ===")
for i, name in enumerate(model.get("feature_names", [])):
    print(f"{i + 1:02d}. {name}")

def inspect_structure(obj, path, depth=0, max_depth=3):
    prefix = "  " * depth

    if isinstance(obj, dict):
        print(f"{prefix}{path}: dict, keys={list(obj.keys())[:30]}")

        if depth < max_depth:
            for key, value in list(obj.items())[:20]:
                if isinstance(value, (dict, list)):
                    inspect_structure(
                        value, f"{path}.{key}", depth + 1, max_depth
                    )
                else:
                    print(
                        f"{prefix}  {key}: "
                        f"{type(value).__name__} = {repr(value)[:100]}"
                    )

    elif isinstance(obj, list):
        print(f"{prefix}{path}: list, length={len(obj)}")

        if obj and depth < max_depth:
            inspect_structure(
                obj[0], f"{path}[0]", depth + 1, max_depth
            )

    else:
        print(f"{prefix}{path}: {type(obj).__name__} = {repr(obj)[:100]}")

print("\n=== STRUKTUR MODEL PER HORIZON ===")

for horizon in ["1", "3", "6", "24"]:
    print(f"\n--- Horizon {horizon} jam ---")

    for target in ["temperature", "rain"]:
        component = model["horizons"][horizon][target]
        inspect_structure(
            component,
            f"horizons.{horizon}.{target}"
        )

=== METADATA MODEL ===
Top-level keys: ['schema_version', 'model_version', 'created_utc', 'trained_location', 'training_period', 'timezone', 'utc_offset_seconds', 'data_source', 'target_definition', 'rain_threshold_mm', 'min_history_hours', 'feature_names', 'refit_all_for_deploy', 'horizons', 'evaluation_summary', 'parity_fixture']
Model version: weather-v2
Model type: None
Jumlah fitur: 38

=== DAFTAR FITUR ===
01. temperature_2m_current
02. relative_humidity_2m_current
03. pressure_msl_current
04. wind_speed_10m_current
05. precipitation_current
06. cloud_cover_current
07. temperature_lag_1
08. temperature_lag_3
09. temperature_lag_6
10. temperature_lag_24
11. humidity_lag_1
12. humidity_lag_3
13. humidity_lag_6
14. humidity_lag_24
15. precipitation_lag_1
16. precipitation_lag_3
17. precipitation_lag_6
18. precipitation_lag_24
19. pressure_lag_1
20. pressure_lag_24
21. hour_sin
22. hour_cos
23. day_of_year_sin
24. day_of_year_cos
25. pressure_change_3
26. pressure_change_6
27. temper

In [11]:

import json

print("=== DEFINISI TARGET ===")
print(json.dumps(
    model.get("target_definition"),
    indent=2,
    ensure_ascii=False
))

print("\n=== KONFIGURASI MODEL ===")
for key in [
    "schema_version",
    "trained_location",
    "training_period",
    "timezone",
    "data_source",
    "rain_threshold_mm",
    "min_history_hours",
    "refit_all_for_deploy"
]:
    print(f"{key}: {model.get(key)}")

print("\n=== PARITY FIXTURE ===")
fixture = model.get("parity_fixture")

if isinstance(fixture, dict):
    print("Keys:", list(fixture.keys()))
elif isinstance(fixture, list):
    print("Jumlah sampel:", len(fixture))
else:
    print("Tipe fixture:", type(fixture).__name__)

print(json.dumps(
    fixture,
    indent=2,
    ensure_ascii=False
)[:8000])

print("\n=== STRUKTUR NODE GBDT ===")

for horizon in ["1", "3", "6", "24"]:
    for target in ["temperature", "rain"]:
        component = model["horizons"][horizon][target]
        tree = component["trees"][0]

        print(
            f"\nHorizon={horizon}, target={target}, "
            f"type={component.get('type')}, "
            f"model={component.get('model_name')}, "
            f"init={component.get('init')}, "
            f"learning_rate={component.get('learning_rate')}"
        )

        print(
            "Urutan fitur sama dengan fitur global:",
            component.get("features") == model["feature_names"]
        )

        for key in ["l", "r", "f", "t", "v"]:
            print(f"{key}:", tree[key][:12])

        leaf_indices = [
            i for i, (left, right) in enumerate(
                zip(tree["l"], tree["r"])
            )
            if left < 0 and right < 0
        ]

        print("Contoh indeks leaf:", leaf_indices[:10])
        print("Jumlah leaf pada pohon pertama:", len(leaf_indices))

=== DEFINISI TARGET ===
"temperature: T(t+h) = T(t) + delta (model memprediksi delta); rain: P(precipitation(t+h) >= rain_threshold_mm) pada jam target"

=== KONFIGURASI MODEL ===
schema_version: 2
trained_location: {'latitude': -7.2575, 'longitude': 112.7521}
training_period: {'start': '2022-01-01', 'end': '2026-10-08'}
timezone: Asia/Jakarta
data_source: Open-Meteo Historical Weather API (reanalysis), CC BY 4.0
rain_threshold_mm: 0.1
min_history_hours: 25
refit_all_for_deploy: True

=== PARITY FIXTURE ===
Jumlah sampel: 3
[
  {
    "history": [
      {
        "time": "2024-05-19T13:00",
        "temperature_2m": 32.2,
        "relative_humidity_2m": 51.0,
        "precipitation": 0.0,
        "pressure_msl": 1006.5,
        "wind_speed_10m": 17.2,
        "cloud_cover": 19.0
      },
      {
        "time": "2024-05-19T14:00",
        "temperature_2m": 32.0,
        "relative_humidity_2m": 52.0,
        "precipitation": 0.0,
        "pressure_msl": 1005.5,
        "wind_speed_10m": 

In [12]:

import json

# 1. Periksa bentuk parity fixture
print("=== PARITY FIXTURE ===")

for i, sample in enumerate(model.get("parity_fixture", [])):
    print(f"\nSampel {i + 1}")
    print("Keys:", list(sample.keys()))

    history = sample.get("history", [])
    print("Jumlah history:", len(history))

    if history:
        print("Awal history:", history[0].get("time"))
        print("Akhir history:", history[-1].get("time"))

    for key, value in sample.items():
        if key == "history":
            continue

        if isinstance(value, dict):
            print(key, "| keys:", list(value.keys()))
            print(json.dumps(value, ensure_ascii=False)[:1200])
        else:
            print(key, "|", repr(value)[:1200])


# 2. Cari formula fitur dan exporter dalam kode
# yang sudah dijalankan pada sesi Colab.
terms = [
    "pressure_change_3",
    "temperature_change_3",
    "precip_sum_24",
    "temperature_range_24",
    "dewpoint_depression_current",
    "hours_since_rain",
    "parity_fixture",
    "learning_rate",
    "trees",
]

print("\n=== SEL KODE YANG RELEVAN ===")

for cell_number, source in enumerate(In):
    if not isinstance(source, str):
        continue

    hits = [term for term in terms if term in source]

    if hits:
        print(
            f"\n--- Input cell {cell_number}; "
            f"cocok: {', '.join(hits)}; "
            f"panjang: {len(source)} karakter ---"
        )

        # Tampilkan cuplikan awal untuk mengidentifikasi
        # fungsi; bukan seluruh kode yang mungkin panjang.
        print(source[:5000])

=== PARITY FIXTURE ===

Sampel 1
Keys: ['history', 'expected']
Jumlah history: 72
Awal history: 2024-05-19T13:00
Akhir history: 2024-05-22T12:00
expected | keys: ['1', '3', '6', '24']
{"1": {"temperature_c": 31.982542952269547, "rain_probability": 0.04268102678452212}, "3": {"temperature_c": 31.43662554725574, "rain_probability": 0.06975125721439912}, "6": {"temperature_c": 29.155822031172914, "rain_probability": 0.09734685572752383}, "24": {"temperature_c": 31.58036981543979, "rain_probability": 0.15910232401965477}}

Sampel 2
Keys: ['history', 'expected']
Jumlah history: 72
Awal history: 2026-05-19T01:00
Akhir history: 2026-05-22T00:00
expected | keys: ['1', '3', '6', '24']
{"1": {"temperature_c": 25.826766520066688, "rain_probability": 0.8260282209927098}, "3": {"temperature_c": 25.50007448522414, "rain_probability": 0.45954112149465426}, "6": {"temperature_c": 25.30114031372222, "rain_probability": 0.2526157315110673}, "24": {"temperature_c": 27.269249357068457, "rain_probability":

In [13]:
import json
from pathlib import Path

model_path = OUT / "weather_model.json"
model = json.loads(model_path.read_text(encoding="utf-8"))

fixtures = model["parity_fixture"]

print("Jumlah fixture:", len(fixtures))
print("Semua fixture memiliki features:",
      all("features" in item for item in fixtures))

for i, item in enumerate(fixtures, start=1):
    print(
        f"Sampel {i}:",
        len(item["features"]),
        "fitur;",
        "horizon:",
        list(item["expected"].keys())
    )

Jumlah fixture: 3
Semua fixture memiliki features: False


KeyError: 'features'